# Group 1 Baseline Workflow (Storyteller Notebook)

This notebook is the **clean orchestration layer** for Group 1 baseline.
All core logic lives in `src/`; notebook cells call those modules stage-by-stage.


## Stage Map

1. Setup and environment check
2. Data prep (COCO -> alignment -> instruction format)
3. Tokenization (stage 1 and stage 2)
4. CLIP feature precompute
5. Manifest build
6. Stage 1 training
7. Stage 2 training
8. Artifacts + quick validation


In [ ]:
from pathlib import Path
import json
import os
import sys

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "configs" / "workflow_paths.json"
DOTENV_PATH = PROJECT_ROOT / ".env"

sys.path.append(str(SRC_DIR))
sys.path.append(str(SRC_DIR / "data_prep"))
sys.path.append(str(SRC_DIR / "vision_features"))
sys.path.append(str(SRC_DIR / "training_manifests"))
sys.path.append(str(SRC_DIR / "training"))
sys.path.append(str(SRC_DIR / "model_internals"))

from config_loader import load_dotenv_file, load_json_config

load_dotenv_file(DOTENV_PATH)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("CONFIG_PATH =", CONFIG_PATH)
print("HF_TOKEN loaded =", bool(os.environ.get("HF_TOKEN")))


In [ ]:
# Load central path config from configs/workflow_paths.json
cfg = load_json_config(CONFIG_PATH, PROJECT_ROOT)

Path(cfg["artifacts_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(cfg, indent=2))


## Stage 0: Environment Check

Run this before heavy steps to confirm required imports are available.


In [ ]:
# Run from `code/group1_baseline` root in terminal if this cell fails:
# python scripts/check_env.py --profile core
# python scripts/check_env.py --profile notebook
# python scripts/check_env.py --profile tpu


## Stage 1: Data Prep (COCO -> Alignment -> Instruction Format)

Inputs:
- `cfg["coco_json"]`

Outputs:
- `cfg["stage1_alignment_json"]`
- `cfg["stage1_chat_json"]`


In [ ]:
from prepare_stage1_dataset import build_alignment
from convert_alignment_format import convert_alignment_rows

coco_path = Path(cfg["coco_json"])
alignment_path = Path(cfg["stage1_alignment_json"])
chat_path = Path(cfg["stage1_chat_json"])

alignment = build_alignment(coco_path)
alignment_path.parent.mkdir(parents=True, exist_ok=True)
alignment_path.write_text(json.dumps(alignment, indent=2), encoding="utf-8")

chat_rows = convert_alignment_rows(alignment, seed=42)
chat_path.parent.mkdir(parents=True, exist_ok=True)
chat_path.write_text(json.dumps(chat_rows, indent=2), encoding="utf-8")

print("alignment rows:", len(alignment))
print("chat rows:", len(chat_rows))
print("sample:", chat_rows[0] if chat_rows else "<empty>")


## Stage 2: Tokenization (Stage 1 + Stage 2)

You must initialize a tokenizer first (same family used in training).


In [ ]:
from tokenization import build_tokenized_stage1_dataset, build_tokenized_stage2_dataset
# from transformers import AutoTokenizer

# Example:
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# build_tokenized_stage1_dataset(tokenizer, cfg["stage1_chat_json"], cfg["stage1_tokenized_json"], max_len=128)
# build_tokenized_stage2_dataset(tokenizer, cfg["stage2_input_json"], cfg["stage2_tokenized_json"], max_len=256)

print("Tokenization hooks loaded. Initialize tokenizer and uncomment calls.")


## Stage 3: CLIP Feature Precompute

Inputs:
- Tokenized stage 1 data
- Raw images

Output:
- `.npy` CLIP embeddings in `cfg["clip_feature_dir"]`


In [ ]:
from clip_helpers import create_clip_from_flax_checkpoint
from clip_features import precompute_clip_features_jitted

# Example:
# clip_bundle = create_clip_from_flax_checkpoint(download_if_missing=True)
# precompute_clip_features_jitted(
#     clip_bundle=clip_bundle,
#     tokenized_json=cfg["stage1_tokenized_json"],
#     image_root=cfg["image_root"],
#     output_dir=cfg["clip_feature_dir"],
# )

print("CLIP precompute hooks loaded. Create clip_bundle and uncomment call.")


## Stage 4: Manifest Build

Build stage-specific manifests that join text tokens with vision feature paths.


In [ ]:
from build_stage1_manifest import build_stage1_manifest
from build_stage2_manifest import build_stage2_manifest

# build_stage1_manifest(cfg["stage1_tokenized_json"], cfg["clip_feature_dir"], cfg["stage1_manifest_json"])
# build_stage2_manifest(cfg["stage2_tokenized_json"], cfg["clip_feature_dir"], cfg["stage2_manifest_json"])

print("Manifest builders loaded. Uncomment calls after tokenization/features exist.")


## Stage 5: Stage 1 Training (Projector Alignment)

This stage trains projector parameters while keeping base LLM frozen.


In [ ]:
from flax import nnx
import optax
import pickle

from projector import VisionProjector
from stage1 import run_stage1_training

# Expected objects before this stage (from your model loading pipeline):
# - llama_model
# - llama_graphdef, llama_state = nnx.split(llama_model)

# Example projector init:
# projector = VisionProjector(in_dim=768, out_dim=2048, rngs=nnx.Rngs(0))
# projector_graphdef, projector_state = nnx.split(projector)
# tx = optax.adamw(learning_rate=1e-4)
# opt_state = tx.init(projector_state)

# projector_state, opt_state = run_stage1_training(
#     manifest_json=cfg["stage1_manifest_json"],
#     projector_state=projector_state,
#     opt_state=opt_state,
#     projector_graphdef=projector_graphdef,
#     llama_state=llama_state,
#     llama_graphdef=llama_graphdef,
#     tx=tx,
#     num_epochs=1,
#     batch_size=2,
# )

# with open(cfg["stage1_projector_state"], "wb") as f:
#     pickle.dump(projector_state, f)

print("Stage 1 training hooks loaded.")


## Stage 6: Stage 2 Training (Multimodal Finetuning)

This stage updates projector + LLM states jointly.


In [ ]:
import pickle
from stage2 import run_stage2_training

# Example restore + run:
# with open(cfg["stage1_projector_state"], "rb") as f:
#     projector_state = pickle.load(f)
#
# # tx for joint updates should be initialized on {"projector": projector_state, "llama": llama_state}
# projector_state, llama_state, opt_state = run_stage2_training(
#     manifest_json=cfg["stage2_manifest_json"],
#     projector_state=projector_state,
#     llama_state=llama_state,
#     opt_state=opt_state,
#     projector_graphdef=projector_graphdef,
#     llama_graphdef=llama_graphdef,
#     tx=tx,
#     num_epochs=1,
#     batch_size=2,
# )
#
# with open(cfg["stage2_projector_state"], "wb") as f:
#     pickle.dump(projector_state, f)
# with open(cfg["stage2_llama_state"], "wb") as f:
#     pickle.dump(llama_state, f)

print("Stage 2 training hooks loaded.")


## Stage 7: Artifacts + Quick Validation

Minimal checks to verify pipeline outputs exist before evaluation.


In [ ]:
from pathlib import Path

checks = [
    cfg["stage1_chat_json"],
    cfg["stage1_tokenized_json"],
    cfg["stage1_manifest_json"],
    cfg["stage2_tokenized_json"],
    cfg["stage2_manifest_json"],
]

for p in checks:
    exists = Path(p).exists()
    print(f"{p} -> {'OK' if exists else 'MISSING'}")
